# Aula 1.3 — MLPClassifier (prática complementar)

**Disciplina:** Programação em Python para IA — IFNMG/Ceadi

Notebook **complementar** à videoaula: aqui você roda o `MLPClassifier` sobre uma amostra do dataset de **triagem de COVID** (e-SUS Notifica), avalia com **acurácia** e revisa a **matriz de confusão** e o **classification_report**.

**No Colab:** faça upload de `triagem-covid-amostra.csv` (arraste para a barra lateral clicando no símbolo de pasta). Rode as células na ordem (Shift+Enter).

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

## 1. Carregar os dados
Cada linha é um caso; as colunas são sintomas, estado, faixa etária, raça e estação (já em 0/1). A coluna `target` é o rótulo: 1 = COVID confirmado, 0 = descartado.

In [2]:
df = pd.read_csv("triagem-covid-amostra-raw.csv")
print("linhas, colunas:", df.shape)
df.head()

linhas, colunas: (4000, 13)


,estado,idade,racacor,estacao,sint_tosse,sint_febre,sint_dor_cabeca,sint_dor_garganta,sint_coriza,sint_dispneia,sint_anosmia_ageusia,cond_respiratoria,target
0,RJ,50,Nao informado,verao,0,0,0,0,0,0,0,0,1
1,SP,24,Parda,outono,0,1,0,1,1,0,0,0,0
2,SP,6,Branca,outono,1,1,0,0,1,0,0,0,0
3,SP,6,Branca,inverno,0,0,0,0,0,0,0,0,0
4,RJ,37,Outra,primavera,0,1,1,0,0,0,0,0,0


### Conhecer os dados
Antes de modelar, veja os **tipos** das colunas e um resumo (`describe`). Aqui todas as colunas já são numéricas (0/1).

In [3]:
print("tipos de coluna:")
print(df.dtypes.value_counts())
print()
df.describe()

tipos de coluna:
int64    10
str       3
Name: count, dtype: int64



,idade,sint_tosse,sint_febre,sint_dor_cabeca,sint_dor_garganta,sint_coriza,sint_dispneia,sint_anosmia_ageusia,cond_respiratoria,target
count,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000,4000.000000
mean,30.951000,0.540750,0.367250,0.420250,0.364000,0.451000,0.096750,0.071750,0.018250,0.474000
std,13.814424,0.498399,0.482116,0.493661,0.481209,0.497655,0.295654,0.258106,0.133871,0.499386
min,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,24.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,37.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,37.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,50.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## 3. Dividir em treino e teste
`train_test_split` separa os dados: o modelo **aprende no treino** e é **avaliado no teste** (dados que ele nunca viu).
- `test_size=0.2`: 20% para teste.
- `stratify=y`: mantém a mesma proporção de classes nos dois conjuntos.
- `random_state=42`: torna a divisão reproduzível (sempre a mesma).

In [4]:
y = df["target"]
X = df.drop(columns=["target"])
print("nº de features:", X.shape[1])

nº de features: 12


## 3. Dividir em treino e teste

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("treino:", X_tr.shape[0], "| teste:", X_te.shape[0])

treino: 3200 | teste: 800


## 4. Criar e treinar o MLPClassifier
A mesma MLP do Módulo 1.2, agora pronta no scikit-learn: camadas ocultas 128 → 16, ativação ReLU.

In [6]:
X_tr.dtypes

estado                    str
idade                   int64
racacor                   str
estacao                   str
sint_tosse              int64
sint_febre              int64
sint_dor_cabeca         int64
sint_dor_garganta       int64
sint_coriza             int64
sint_dispneia           int64
sint_anosmia_ageusia    int64
cond_respiratoria       int64
dtype: object

In [17]:
clf = MLPClassifier(
    hidden_layer_sizes=(128, 16),
    activation="relu",
    max_iter=300,
    early_stopping=True,
    random_state=42,
)
clf.fit(X_tr, y_tr)

ValueError: could not convert string to float: 'MG'

## 5. Prever a classe
`predict` devolve a classe final (0 ou 1) para cada caso de teste.

In [ ]:
y_pred = clf.predict(X_te)
print("5 primeiras previsões:", y_pred[:5])

## 6. Avaliar com acurácia
Acurácia = proporção de previsões corretas. É a métrica mais simples de ler.

In [ ]:
acc = accuracy_score(y_te, y_pred)
print("Acurácia:", round(acc, 3))

## 7. Revisão: matriz de confusão e classification_report
Você já viu essas métricas em Inteligência Artificial. A **matriz de confusão** mostra acertos e erros por classe; o **classification_report** traz precision, recall e F1 de cada classe.

In [ ]:
cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=clf.classes_)
disp.plot()
plt.show()
print(classification_report(y_te, y_pred, digits=2))

## 8. Para pensar
A acurácia ficou em torno de 0,6. Olhando a matriz de confusão, o modelo erra mais em qual classe? Por que só a acurácia pode não contar a história toda?